# Huawei-Inspired Healthcare Machine Learning Lab 5
## K-Means Clustering for Exploring Heart-Failure Patient Profiles

**Healthcare task:** Use K-means to explore groups of records with similar ejection-fraction and serum-creatinine measurements.

**Official dataset:** UCI Heart Failure Clinical Records

- Dataset page: https://archive.ics.uci.edu/dataset/519/heart+failure+clinical+records
- Dataset DOI: https://doi.org/10.24432/C5Z89R
- Licence: CC BY 4.0

This notebook adapts the Huawei HCIA-AI V4.0 K-means experiment. The Huawei lab visualises two-dimensional data, groups it first into three clusters and then into four clusters, displays the cluster labels and marks the centroids. This healthcare version follows the same sequence using public clinical-record data.

**Educational use only:** The clusters are mathematical groups, not diagnoses, disease stages or treatment recommendations.

## How to read this notebook

- A line beginning with `#` is a comment for you. Python ignores it.
- Run the cells from top to bottom.
- **Clustering** means grouping similar records without giving the model the correct answer beforehand.
- **K** is the number of groups we ask the model to create.
- A **centroid** is the mathematical centre of a cluster.
- Cluster numbers such as 0, 1 and 2 are arbitrary names. Cluster 2 is not automatically more severe than Cluster 1.

## Learning objectives

By the end of this lab, learners should be able to:

1. Explain the difference between supervised and unsupervised learning.
2. Load a public healthcare dataset.
3. Select numerical features for clustering.
4. Standardise features before using K-means.
5. Create and visualise three-cluster and four-cluster solutions.
6. Interpret cluster centroids and summary tables.
7. Use inertia, the elbow method and silhouette score carefully.
8. Explain why clusters must not be presented as clinical diagnoses.

## Step 1 — Install and import the required packages

In [ ]:
# Install the helper package that downloads datasets from the official UCI repository.
!pip -q install ucimlrepo

# NumPy helps us work with numerical arrays.
import numpy as np
# Pandas helps us work with tables, similar to an Excel sheet.
import pandas as pd
# Matplotlib helps us draw charts.
import matplotlib.pyplot as plt

# This function downloads a dataset from UCI.
from ucimlrepo import fetch_ucirepo
# StandardScaler makes features use a comparable numerical scale.
from sklearn.preprocessing import StandardScaler
# KMeans is the clustering algorithm used in this lab.
from sklearn.cluster import KMeans
# Silhouette score helps us assess how separated the clusters are.
from sklearn.metrics import silhouette_score
# PCA helps us display a multi-feature clustering result in two dimensions.
from sklearn.decomposition import PCA

# A fixed random-state number makes the results repeatable.
RANDOM_STATE = 42
print('Packages imported successfully.')

## Step 2 — Download the healthcare dataset

The UCI dataset contains medical records for 299 people with heart failure. It includes 12 input features and a separate outcome variable.

K-means is unsupervised, so we will **not** give the outcome variable to the clustering model.

In [ ]:
# Download UCI dataset number 519.
dataset = fetch_ucirepo(id=519)

# Copy only the input-feature table.
# The separate target table is deliberately not used for clustering.
features = dataset.data.features.copy()

# Convert all feature values into numbers.
features = features.apply(pd.to_numeric, errors='coerce')

print('Dataset name:', dataset.metadata.name)
print('Number of records:', features.shape[0])
print('Number of available input features:', features.shape[1])
print('Feature names:', list(features.columns))
features.head()

## Step 3 — Select two features for the first clustering experiment

To mirror Huawei's two-dimensional demonstration, we use:

- `ejection_fraction`: percentage of blood leaving the heart during each contraction
- `serum_creatinine`: serum-creatinine concentration in mg/dL

Using two features lets us see the records, clusters and centroids directly on one chart.

In [ ]:
# List the two features used in the visual experiment.
selected_features = ['ejection_fraction', 'serum_creatinine']

# Check that both columns exist in the downloaded dataset.
missing_features = [
    column for column in selected_features
    if column not in features.columns
]

# Stop with a clear message if the expected columns are unavailable.
if missing_features:
    raise ValueError(f'Missing expected columns: {missing_features}')

# Keep only the selected columns and remove incomplete rows.
cluster_data = features[selected_features].dropna().copy()

print('Records available for clustering:', len(cluster_data))
cluster_data.head()

## Step 4 — Examine the selected measurements

In [ ]:
# Show the average, spread, minimum and maximum of each measurement.
cluster_data.describe().round(2)

In [ ]:
# Draw every record before clustering.
# At this stage, all points have the same appearance because no clusters exist yet.
plt.figure(figsize=(8, 6))
plt.scatter(
    cluster_data['ejection_fraction'],
    cluster_data['serum_creatinine'],
    alpha=0.65,
    s=35
)
plt.xlabel('Ejection fraction (%)')
plt.ylabel('Serum creatinine (mg/dL)')
plt.title('Healthcare Records Before Clustering')
plt.show()

## Step 5 — Standardise the measurements

**Why this matters:** Ejection fraction and serum creatinine use different units and numerical ranges. Without standardisation, the feature with the larger numerical spread could influence the distance calculation more strongly.

Standardisation changes the scale for the mathematics but does not change the order of the records.

In [ ]:
# Create the scaling tool.
scaler = StandardScaler()

# Learn the mean and spread of each feature, then standardise the data.
X_scaled = scaler.fit_transform(cluster_data)

# Put the standardised values into a table for easy viewing.
scaled_table = pd.DataFrame(
    X_scaled,
    columns=['standardised_ejection_fraction', 'standardised_serum_creatinine']
)

scaled_table.head()

## Step 6 — Perform K-means clustering with three clusters

This follows the Huawei experiment by first asking K-means to produce three groups.

In [ ]:
# Ask K-means to create three clusters.
k_three = 3
model_three = KMeans(
    n_clusters=k_three,
    random_state=RANDOM_STATE,
    # Run the algorithm several times and keep the best solution.
    n_init=10
)

# Fit the model and obtain one cluster number for every record.
labels_three = model_three.fit_predict(X_scaled)

# The centroids are initially on the standardised scale.
centroids_three_scaled = model_three.cluster_centers_
# Convert them back to the original clinical units for easier interpretation.
centroids_three_original = scaler.inverse_transform(centroids_three_scaled)

print('First 20 cluster labels:')
print(labels_three[:20])

## Step 7 — Visualise the three-cluster result

Each point is one record. Points with the same marker group belong to the same mathematical cluster. The large `X` symbols show the cluster centroids.

In [ ]:
# Draw one group at a time.
plt.figure(figsize=(9, 7))
for cluster_number in range(k_three):
    cluster_mask = labels_three == cluster_number
    plt.scatter(
        cluster_data.loc[cluster_mask, 'ejection_fraction'],
        cluster_data.loc[cluster_mask, 'serum_creatinine'],
        s=40,
        alpha=0.70,
        label=f'Cluster {cluster_number}'
    )

# Mark the centroid of each cluster with a large X.
plt.scatter(
    centroids_three_original[:, 0],
    centroids_three_original[:, 1],
    marker='X',
    s=260,
    edgecolors='black',
    linewidths=1.5,
    label='Centroids'
)

plt.xlabel('Ejection fraction (%)')
plt.ylabel('Serum creatinine (mg/dL)')
plt.title('K-Means Result with Three Clusters')
plt.legend()
plt.show()

## Step 8 — Display the three centroids in original units

The centroid is the average location of the cluster in the two-dimensional feature space. It is not necessarily a real person and must not be interpreted as an ideal or typical patient.

In [ ]:
# Create a readable centroid table.
centroid_table_three = pd.DataFrame(
    centroids_three_original,
    columns=['Ejection fraction (%)', 'Serum creatinine (mg/dL)']
)
centroid_table_three.insert(0, 'Cluster', range(k_three))
centroid_table_three.round(2)

## Step 9 — Summarise the three clusters

In [ ]:
# Add the cluster labels to a fresh copy of the original two-feature data.
profile_three = cluster_data.copy()
profile_three['cluster'] = labels_three

# Calculate how many records are in each cluster and the average measurements.
summary_three = profile_three.groupby('cluster').agg(
    number_of_records=('cluster', 'size'),
    mean_ejection_fraction=('ejection_fraction', 'mean'),
    mean_serum_creatinine=('serum_creatinine', 'mean')
).reset_index()

summary_three.round(2)

## Step 10 — Repeat the experiment with four clusters

The Huawei lab then changes the number of clusters. Here we repeat the healthcare experiment with K = 4.

In [ ]:
# Ask K-means to create four clusters.
k_four = 4
model_four = KMeans(
    n_clusters=k_four,
    random_state=RANDOM_STATE,
    n_init=10
)

# Fit the model and obtain labels.
labels_four = model_four.fit_predict(X_scaled)
# Convert the centroids back to original clinical units.
centroids_four_original = scaler.inverse_transform(model_four.cluster_centers_)

print('First 20 four-cluster labels:')
print(labels_four[:20])

## Step 11 — Visualise the four-cluster result

In [ ]:
# Draw one group at a time.
plt.figure(figsize=(9, 7))
for cluster_number in range(k_four):
    cluster_mask = labels_four == cluster_number
    plt.scatter(
        cluster_data.loc[cluster_mask, 'ejection_fraction'],
        cluster_data.loc[cluster_mask, 'serum_creatinine'],
        s=40,
        alpha=0.70,
        label=f'Cluster {cluster_number}'
    )

# Mark the four centroids.
plt.scatter(
    centroids_four_original[:, 0],
    centroids_four_original[:, 1],
    marker='X',
    s=260,
    edgecolors='black',
    linewidths=1.5,
    label='Centroids'
)

plt.xlabel('Ejection fraction (%)')
plt.ylabel('Serum creatinine (mg/dL)')
plt.title('K-Means Result with Four Clusters')
plt.legend()
plt.show()

## Step 12 — Summarise the four clusters

In [ ]:
# Add the four-cluster labels to a fresh copy of the data.
profile_four = cluster_data.copy()
profile_four['cluster'] = labels_four

# Calculate a descriptive summary for each group.
summary_four = profile_four.groupby('cluster').agg(
    number_of_records=('cluster', 'size'),
    mean_ejection_fraction=('ejection_fraction', 'mean'),
    mean_serum_creatinine=('serum_creatinine', 'mean')
).reset_index()

summary_four.round(2)

## Step 13 — Compare the three-cluster and four-cluster solutions

- **Inertia** measures how close records are to their own centroid. Lower is better, but inertia always tends to decrease when K increases.
- **Silhouette score** ranges approximately from -1 to 1. Higher values usually indicate more clearly separated clusters.

These measures help describe the mathematical structure. They do not prove clinical usefulness.

In [ ]:
# Calculate silhouette scores for both solutions.
silhouette_three = silhouette_score(X_scaled, labels_three)
silhouette_four = silhouette_score(X_scaled, labels_four)

# Place the comparison in a table.
comparison_table = pd.DataFrame({
    'Number of clusters': [3, 4],
    'Inertia': [model_three.inertia_, model_four.inertia_],
    'Silhouette score': [silhouette_three, silhouette_four]
})

comparison_table.round(3)

## Step 14 — Use the elbow method to explore K

The elbow method fits several models and plots inertia against K. We look for a point where the improvement begins to slow, like a bend in an arm.

The elbow may be unclear, so it should not be treated as an automatic answer.

In [ ]:
# Try K values from 2 to 8.
k_values = range(2, 9)
inertia_values = []
silhouette_values = []

for k_value in k_values:
    # Train one K-means model for this value of K.
    temporary_model = KMeans(
        n_clusters=k_value,
        random_state=RANDOM_STATE,
        n_init=10
    )
    temporary_labels = temporary_model.fit_predict(X_scaled)

    # Save both evaluation values.
    inertia_values.append(temporary_model.inertia_)
    silhouette_values.append(silhouette_score(X_scaled, temporary_labels))

k_evaluation = pd.DataFrame({
    'K': list(k_values),
    'Inertia': inertia_values,
    'Silhouette score': silhouette_values
})

k_evaluation.round(3)

In [ ]:
# Draw the elbow chart.
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertia_values, marker='o')
plt.xlabel('Number of clusters, K')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.xticks(list(k_values))
plt.show()

In [ ]:
# Draw the silhouette-score chart.
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), silhouette_values, marker='o')
plt.xlabel('Number of clusters, K')
plt.ylabel('Silhouette score')
plt.title('Silhouette Score for Different K Values')
plt.xticks(list(k_values))
plt.show()

## Step 15 — Assign a fictional new record to a cluster

K-means does not predict a diagnosis. It assigns a new record to the nearest mathematical centroid.

In [ ]:
# Create one fictional record using the same two feature names.
fictional_record = pd.DataFrame([{
    'ejection_fraction': 35,
    'serum_creatinine': 1.4
}])

# Standardise it using the scaler learned from the original dataset.
fictional_scaled = scaler.transform(fictional_record)
# Assign it to the nearest cluster in the three-cluster solution.
fictional_cluster = model_three.predict(fictional_scaled)[0]

print('Assigned mathematical cluster:', fictional_cluster)
print('This output is not a diagnosis or risk category.')

## Optional extension — Cluster using five features

The first experiment used two features so the clusters could be drawn directly. In this extension, we use five continuous features:

- age
- ejection fraction
- serum creatinine
- serum sodium
- platelets

PCA then compresses the five standardised dimensions into two display axes. The PCA axes are mathematical combinations, not clinical measurements.

In [ ]:
# Select five continuous features.
five_features = [
    'age',
    'ejection_fraction',
    'serum_creatinine',
    'serum_sodium',
    'platelets'
]

# Keep complete rows only.
five_feature_data = features[five_features].dropna().copy()

# Standardise the five features.
five_feature_scaler = StandardScaler()
five_feature_scaled = five_feature_scaler.fit_transform(five_feature_data)

# Create three clusters using all five standardised features.
five_feature_model = KMeans(
    n_clusters=3,
    random_state=RANDOM_STATE,
    n_init=10
)
five_feature_labels = five_feature_model.fit_predict(five_feature_scaled)

print('Five-feature silhouette score:', round(
    silhouette_score(five_feature_scaled, five_feature_labels),
    3
))

In [ ]:
# Reduce the five standardised features to two display dimensions.
pca = PCA(n_components=2)
pca_coordinates = pca.fit_transform(five_feature_scaled)

# Draw the PCA representation of the three clusters.
plt.figure(figsize=(9, 7))
for cluster_number in range(3):
    cluster_mask = five_feature_labels == cluster_number
    plt.scatter(
        pca_coordinates[cluster_mask, 0],
        pca_coordinates[cluster_mask, 1],
        s=40,
        alpha=0.70,
        label=f'Cluster {cluster_number}'
    )

plt.xlabel('PCA display axis 1')
plt.ylabel('PCA display axis 2')
plt.title('Five-Feature Clustering Shown with PCA')
plt.legend()
plt.show()

print('Variation shown by the two display axes:',
      round(pca.explained_variance_ratio_.sum(), 3))

In [ ]:
# Add cluster numbers to the five-feature table.
five_feature_profiles = five_feature_data.copy()
five_feature_profiles['cluster'] = five_feature_labels

# Calculate the number of records and average feature values in each cluster.
five_feature_summary = five_feature_profiles.groupby('cluster').agg(
    number_of_records=('cluster', 'size'),
    mean_age=('age', 'mean'),
    mean_ejection_fraction=('ejection_fraction', 'mean'),
    mean_serum_creatinine=('serum_creatinine', 'mean'),
    mean_serum_sodium=('serum_sodium', 'mean'),
    mean_platelets=('platelets', 'mean')
).reset_index()

five_feature_summary.round(2)

## Student exercises

1. Change K from 3 to 2, 4, 5 and 6.
2. Compare the inertia and silhouette score for each K.
3. Replace serum creatinine with age in the two-feature experiment.
4. Explain why standardisation is important for distance-based algorithms.
5. Run the model with a different `random_state` and compare the labels.
6. Compare the two-feature and five-feature cluster summaries.
7. Explain why calling a cluster “high risk” would require additional clinical validation.

## Responsible-use notes

- K-means always creates the requested number of clusters, even when the data do not contain clinically meaningful groups.
- Cluster numbers are arbitrary and can change between model runs.
- K-means uses distance and can be affected by unusual values and feature scaling.
- A cluster is not a diagnosis, phenotype, disease stage or treatment recommendation.
- Do not enter identifiable patient information into this notebook.
- Clinical research would require a prespecified protocol, expert interpretation, stability testing, external validation, fairness assessment and ethics approval.

## Dataset citation

*Heart Failure Clinical Records* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5Z89R